# 恢复输入域：固定模型与门槛，不拟合
读取训练标准化参数，复现所有既有合格日分数并做精确logit分解。z值仅描述偏移，不按3σ删除数据，不按收益选择特征。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.workflow.stage1_open_action_value import prepare_on_policy_recovery
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec,FEATURE_COLUMNS,_feature_row
from scipy.special import expit
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/recovery_input_domain_20261014';plan=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013';qlib.init(provider_uri=str(provider),region='us')
accounts,account_audit=_source(root,plan['account_source'],['inputs/frame',*[f'analysis/native/cash_start_2023_current_anchor_{c}' for c in plan['costs']]])
action,action_audit=_source(root,plan['action_source'],['inputs/plan','controller/source_recovery_selection']);model_source,model_audit=_source(root,plan['controller_source'],['inputs/controller'])
selection=action['controller/source_recovery_selection'];spec=IntradayRecoveryDirectAllocationSpec(root=root,**action['inputs/plan']['recovery_spec']);model=model_source['inputs/controller']['frozen_recovery_model']
release=json.loads((root/'production/stage1_futu_sim_v1/runtime_release.json').read_text());live_summary=json.loads((root/release['action']['card_path']/'summary.json').read_text());assert hashlib.sha256((root/release['action']['card_path']/'summary.json').read_bytes()).hexdigest()==release['action']['summary_sha256']
assert live_summary['source_stage1']['selected_model']['minimum_source_alpha_probability']==selection['minimum_source_alpha_probability']
frame=accounts['inputs/frame'].set_index('trade_date',drop=False);funnel=[];records={};profiles=[];contributions={};raw_inputs={}
scaler=model.named_steps['standardscaler'];classifier=model.named_steps['logisticregression'];imputer=model.named_steps['simpleimputer']
assert len(FEATURE_COLUMNS)==len(scaler.mean_)==len(classifier.coef_[0])
for cost in plan['costs']:
 book=accounts[f'analysis/native/cash_start_2023_current_anchor_{cost}'];ledger=book['independent_replay'];previous_q=np.zeros(2);previous_cash=plan['initial_cash'];previous_nav=None;history=[];daily=[];inputs=[]
 for row in ledger.itertuples(index=False):
  source=frame.loc[pd.Timestamp(row.trade_date)];known=source[['known_soxl','known_soxx']].to_numpy(float);nav=float(previous_q@known+previous_cash)
  if previous_nav is not None:history.append(nav/previous_nav-1)
  previous_nav=nav;raw_beta=3*source.soxl_weight+source.soxx_weight;target=min(raw_beta,selection['action_beta'])
  market_ready=bool(model is not None and pd.notna(source.early_ret_soxx))
  risk_ready=bool(market_ready and row.portfolio_drawdown>=spec.recovery_eligibility_drawdown)
  cap_ready=bool(risk_ready and target>row.current_beta_cap+1e-12)
  alpha_ready=bool(cap_ready and pd.notna(source.alpha_probability) and source.alpha_probability>=selection['minimum_source_alpha_probability'])
  opened={'raw_beta':raw_beta,'drawdown':row.portfolio_drawdown,'current_beta_cap':row.current_beta_cap,'risk_state':row.risk_state,'history':history}
  result=prepare_on_policy_recovery(source,open_state=opened,recovery_spec=spec,model=model,selected_model=selection)
  assert bool(result['recovery_action'])==bool(row.recovery_action)
  if alpha_ready:
   inputs.append({'trade_date':row.trade_date,**_feature_row(source,drawdown=opened['drawdown'],cap=opened['current_beta_cap'],raw_beta=raw_beta,state=opened['risk_state'],history=history)})
  daily.append({'trade_date':row.trade_date,'market_ready':market_ready,'risk_ready':risk_ready,'cap_ready':cap_ready,'alpha_ready':alpha_ready,'recovery_probability':result['recovery_probability'],'recovery_action':result['recovery_action'],'source_alpha':source.alpha_probability})
  previous_q=np.array([row.ending_soxl_quantity,row.ending_soxx_quantity]);previous_cash=row.ending_cash
 data=pd.DataFrame(daily);records[str(cost)]=data
 matrix=pd.DataFrame(inputs).set_index('trade_date')[list(FEATURE_COLUMNS)]
 assert matrix.notna().all().all()
 standardized=scaler.transform(imputer.transform(matrix))
 parts=standardized*classifier.coef_[0]
 logits=parts.sum(axis=1)+classifier.intercept_[0]
 np.testing.assert_allclose(expit(logits),data.set_index('trade_date').loc[matrix.index,'recovery_probability'],rtol=1e-12,atol=1e-12)
 assert matrix[['early_range_soxl','early_range_soxx']].le(0).all().all()
 contributions[str(cost)]=pd.DataFrame(parts,index=matrix.index,columns=FEATURE_COLUMNS);raw_inputs[str(cost)]=matrix
 for i,feature in enumerate(FEATURE_COLUMNS):
  profiles.append({'slippage_bps':cost,'feature':feature,'eligible_rows':len(matrix),'training_mean':float(scaler.mean_[i]),'training_scale':float(scaler.scale_[i]),'current_mean':float(matrix[feature].mean()),'current_min':float(matrix[feature].min()),'current_max':float(matrix[feature].max()),'mean_z':float(standardized[:,i].mean()),'max_absolute_z':float(np.abs(standardized[:,i]).max()),'fraction_absolute_z_gt3':float((np.abs(standardized[:,i])>3).mean()),'coefficient':float(classifier.coef_[0,i]),'mean_logit_contribution':float(parts[:,i].mean())})

 funnel.append({'slippage_bps':cost,'sessions':len(data),'market_ready':int(data.market_ready.sum()),'risk_ready':int(data.risk_ready.sum()),'cap_ready':int(data.cap_ready.sum()),'alpha_ready':int(data.alpha_ready.sum()),'alpha_veto_given_cap_ready':int((data.cap_ready&~data.alpha_ready).sum()),'recovery_actions':int(data.recovery_action.sum()),'minimum_eligible_recovery_probability':float(data.recovery_probability.min()),'maximum_eligible_recovery_probability':float(data.recovery_probability.max())})
summary={'experiment':card.name,'decision':'recovery_input_distribution_diagnostic_no_threshold_change','new_fits':0,'new_backtests':0,'new_orders':0,'parameter_searches':0,'objective_achieved':False,'scaler_training_samples':int(scaler.n_samples_seen_),'logistic_intercept':float(classifier.intercept_[0]),'threshold_log_odds':float(np.log(selection['probability_threshold']/(1-selection['probability_threshold']))),'feature_missing_values':0,'range_sign_contract_verified':True,'recovery_alpha_floor':selection['minimum_source_alpha_probability'],'recovery_probability_threshold':selection['probability_threshold'],'recovery_action_beta':selection['action_beta'],'tail_used_as_direct_cash_probability':False,'tail_alpha_enter_learned_open_and_recovery_features':True,'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/recorder.html','scope':'Conditional inference on saved independent cost-path states; no changed actions, holdings or PnL. Not causal value of removing a gate, not fresh paper evidence.','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily','analysis/raw_inputs','analysis/logit_contributions','sources/*']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='frozen_output_route_audit',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0,new_orders=0)
 rec.save_objects(artifact_path='inputs',plan=plan,account_audit=account_audit,action_audit=action_audit,model_audit=model_audit,selection=selection)
 rec.save_objects(artifact_path='analysis',daily=records,funnel=pd.DataFrame(funnel),input_profiles=pd.DataFrame(profiles),logit_contributions=contributions,raw_inputs=raw_inputs,summary=summary)
 for name in ['stage1_runtime_release','stage1_open_action_value','intraday_recovery_direct_allocation']:
  rec.save_objects(artifact_path='sources',**{name:(root/'src/workflow'/f'{name}.py').read_text()})
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');pd.DataFrame(funnel).to_csv(card/'recovery_funnel.csv',index=False)
pd.DataFrame(profiles).to_csv(card/'input_profiles.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256'];print(pd.DataFrame(funnel).to_string(index=False))


# 恢复模型输入域：没有符号翻转，存在状态与量价分布差异

本轮0拟合、0新账户回测、0参数搜索。对上一轮2/5bp路径中228/476个前置合格日，读取冻结Pipeline的imputer、StandardScaler和LogisticRegression；逐项恢复标准化输入及线性logit贡献，sigmoid后与原冻结恢复分数在1e-12容差内一致。没有用未来收益筛字段或改变门槛。

标准化器记录595个训练样本；截距0.00142582，固定0.825门槛对应log-odds约1.5506。当前输入无缺失，不依赖imputer填值。early_range定义为low/high-1，原训练均值与当前输入均为负，未发现把正振幅错误喂入负振幅模型的问题；不能因为字段名字是range就反转符号。

2bp合格日均值：SOXL log_volume由训练均值15.9325到16.8737（+1.127σ），对应平均logit贡献-0.1656；账户回撤由22.49%到16.32%（-1.538σ），贡献-0.1353；原始SOXX权重由45.28%到35.59%（-1.258σ），贡献-0.1032。5bp中成交量贡献-0.1779，原始SOXX权重-0.0717。它们描述的是已保存模型在当前域的输出分解，不等于去掉某项后的策略收益。

极端首小时振幅达到训练标准差约12.3倍；SOXX振幅|z|>3比例在2/5bp合格域分别3.95%/1.89%。这不等于数据错误，也不是正态假设下显著性检验；极端日保留，不按结果删除或截断。训练均值/标准差不是训练极值范围，本卡未声称当前所有点在训练支持域之外。

原初训练价格面及完整样本分布没有在本卡重建。绝对成交量差异可能包含真实交易活跃度、股份单位/调整基准、年份与资格条件变化，目前不能归因于拆股或坏数据。研究价格已修复不等于冻结训练标准化自动更新，亦不允许悄悄重估scaler。

另一个需明确的迁移边界：源recovery_spec记录teacher_recovery_beta=1.2，而选择的action_beta=0.6。训练“是否值得恢复”的参考动作幅度与实际恢复动作不同；这种设计不自动构成bug，但分类分数不是当前0.6动作的已校准净收益概率。需要另行检查标签/执行域一致性，不能凭0次动作降低0.825门槛。

决定：recovery_input_distribution_diagnostic_no_threshold_change。最高分低于门槛的现象已复现，未证明修复输入或放宽门槛能产生超额。下一步优先核对训练参考动作与实际动作的经济标签，而非仅因大z值进行截断或按本期均值重新标准化。现役release未改，稳定净超额未实现。

[所有21项输入域与精确贡献](input_profiles.csv) · [门槛计数](recovery_funnel.csv)

Recorder `1f8a990078db4ec38b80411c8f712878`。
